# External dataset audit

Count carrot, cabbage and potato reference parcels or classified pixels using local files. Run from the repo root or notebooks folder using the project environment. No Earth Engine access is required. All data paths can be changed below.

- **CropHarvest** uses the pinned [Zenodo label release](https://zenodo.org/records/10251170). The full audit gives 552 distinct screened parcels with an interior 10 m pixel, including 221 JECAM parcels. Imagery and window quality are evaluated separately.
- **EuroCrops v2.0** uses [annual GSA GeoParquet files](https://jeodpp.jrc.ec.europa.eu/ftp/public/JRC-OpenData/DRLL/EuroCropsV2/) named `<NUTS>_<YEAR>.parquet` and the [official mapping CSV](https://github.com/Martincccc/EuroCropsV2/blob/main/data/cropcodemapping/eurocrops.csv). Counts are distinct parcel IDs per region and year. Review the helper's exact translated-name aliases before treating a zero as absence.
- **USDA CDL** uses [official classified raster codes](https://developers.google.com/earth-engine/datasets/catalog/USDA_NASS_CDL). Counts are classified pixels per raster. Field boundaries are needed to count parcels.

In [1]:
from pathlib import Path
import sys
from tempfile import mkdtemp

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
sys.path.insert(0, str(root / "src"))

from crop_pipeline.sources.dataset_audit import (
    audit_cropharvest, audit_eurocrops, audit_usda_cdl,
)

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file())
assert (root / "pyproject.toml").is_file(), "Start from the repository root or notebooks folder"
data = root / "data/external"
labels = data / "cropharvest/labels.geojson"
eurocrops = data / "eurocrops_v2"
mapping = eurocrops / "eurocrops.csv"
usda = data / "usda_cdl"
# Save each audit separately; previous audits and downloads remain intact.
audit_root = root / "pipeline_outputs/collection/reports"
audit_root.mkdir(parents=True, exist_ok=True)
audit_folder = Path(mkdtemp(prefix="reference_audit_", dir=audit_root))


## Dataset roles and saved selections

Sen4AgriNet supplies mostly potato and some carrot; FranceCrops supplies potato/cabbage; EuroCrops adds carrot. Eligible CropHarvest JECAM supplies external testing for all three crops. Current crop-specific requests are top-ups, not the full dataset scope. Counts below describe existing selections, not the full reference releases. USDA remains an exploratory inventory, outside the active experiment.


In [ ]:
from crop_pipeline.collection_summary import inspect_collection

saved_collection = inspect_collection(root)
for source, role in saved_collection["source_roles"].items():
    print(f"{source}: {role}")
from crop_pipeline.notebook_display import show_table
show_table(saved_collection["profile"], title="Saved selection coverage (not imagery usability)")
print("Saved metadata issues:", len(saved_collection["issues"]))


## CropHarvest

In [2]:
if labels.is_file():
    total, counts, by_source = audit_cropharvest(labels, audit_folder / "cropharvest")
    print(f"Full release: {total:,} label records")
    print(counts.to_string())
    print(by_source.to_string(index=False))
else:
    print(f"Place the pinned CropHarvest labels.geojson at {labels}")

Full release: 113,893 label records
         candidates  passed  excluded  raw_alias_matches  interior_pixel_parcels
crop                                                                            
carrot          522     522         0                622                     105
cabbage         190     190         0                265                      91
potato          670     670         0                780                     356
   crop            source  upstream_test  interior_pixel_parcels
cabbage     ile-de-france          False                      14
cabbage             jecam          False                      56
cabbage martinique-france          False                      16
cabbage    reunion-france          False                       5
 carrot     ile-de-france          False                      27
 carrot             jecam          False                      73
 carrot martinique-france          False                       3
 carrot    reunion-france          Fals

## EuroCrops v2.0

In [3]:
files = sorted(eurocrops.rglob("*.parquet"))
if mapping.is_file() and files:
    counts = audit_eurocrops(files, mapping)
    if counts.empty:
        print("No annual GSA files found. Expected filenames such as PT_2021.parquet.")
    else:
        print(counts.pivot(index=["region", "year"], columns="crop", values="declared_parcels").to_string())
else:
    print(f"Place annual GeoParquets in {eurocrops} and the mapping CSV at {mapping}")

crop         cabbage  carrot  potato
region year                         
NL     2021        0       0       0


## USDA Cropland Data Layer

In [4]:
files = sorted(path for path in usda.rglob("*") if path.suffix.lower() in {".tif", ".tiff"})
if files:
    for path in files:
        print(path.relative_to(usda))
        print(audit_usda_cdl(path).to_string())
else:
    print(f"Place USDA CDL GeoTIFFs in {usda}")

CDL_2023_06.tif
crop
carrot     126081
cabbage     15565
potato      67421


## Organized outputs

Refresh the linked output folders without moving downloads or changing saved jobs. Run **06_dataset_summary.ipynb** to record all saved batches and selections. Historical outputs above were retained from the original notebook; rerun deliberately to update them.


In [ ]:
from crop_pipeline.output_layout import refresh

output_layout = refresh(root)
print("Organized collection outputs:", root / "pipeline_outputs/collection")
print("Next: data_collection/06_dataset_summary.ipynb, then preprocessing/01_imagery_quality.ipynb.")
